# PA operating point — ADRV9026

TX1 into ORx1 on LO1. Start at an attenuation where the PA barely compresses.
The ORx AGC levels the capture there, then `find_compression_point` lowers the
attenuation until the PAPR compression on a 0.1 ms window around the aligned
peak is within tolerance of the target. That attenuation is the lock
(backoff 0 dB).

`LOCK_ON` picks what the lock is: `"papr"` (that PAPR compression, the default) or
`"gain"` (the gain compression at the signal peaks, for example 4 dB). Every search step
prints both and the AM/AM top slope; `MIN_TOP_SLOPE` (for example 0.08) makes the search back
off before the PA output flattens at the peaks.

At the lock, one capture is aligned to the reference and saved with one CSV row.
Metrics and columns: `docs/dpd_workflow_spec.md`.

## 1. Parameters

In [ ]:
from adrvtrx import TxChannel, RxChannel

PROFILE = "ADRV9025Init_StdUseCase98_LinkSharing.profile"

TX_CHANNEL  = TxChannel.TX1
ORX_CHANNEL = RxChannel.ORX1

LO1_HZ = 2_400_000_000

SIGNAL_PATH = "new100MHz256QAM_CFRed.txt"
BW_MHZ      = 100

# Safe start (more attenuation = less power). The floor is the hottest
# attenuation the search may set.
TX_ATTEN_START_DB = 20.0
TX_ATTEN_MIN_DB   = 9.0
COARSE_STEP_DB    = 5.0
FINE_STEP_DB      = 0.2

# What the search locks on: "papr" = PAPR compression on the peak window (e.g. 3.0 dB),
# "gain" = gain compression at the signal peaks (e.g. 4.0 dB with COMP_TOL_DB = 0.2).
LOCK_ON               = "papr"
TARGET_COMPRESSION_DB = 3.0
COMP_TOL_DB           = 0.1
WINDOW_MS             = 0.1
MIN_TOP_SLOPE         = None     # clip guard, e.g. 0.08: back off before the PA output goes flat

ORX_TARGET_DBFS = -1.0
ORX_TOL_UP_DB   = 0.3
ORX_TOL_DOWN_DB = 0.6

OUTPUT_OVERSAMPLE = 2        # capture two periods so one aligned period can be cropped
SAVE_DIR = "captures/operating_point"
CSV_NAME = "TX1_conditions.csv"

## 2. Imports, config, profile, signal

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.waveform import full_scale

from adrvtrx.waveform import load_tab_iq, prepare_tx
from adrvtrx.transmit import transmit_bands
from adrvtrx.compression import find_compression_point
from adrvtrx.conditions import (
    ConditionLog, OperatingCondition, capture_point, condition_name, save_aligned_iq,
)
from adrvtrx.metrics import PA_CLIP_SLOPE, papr_db

cfg = load_config()
cfg.profile_name = PROFILE
cfg.lo.lo1_hz = int(LO1_HZ)

info = read_profile(cfg.profile_path)
FS = info.orx_rate_hz
print("Profile:", cfg.profile_path.name)
print(f"TX  {info.tx_rate_khz/1000:.2f} MSPS, {info.tx_bits} bit (FS {full_scale(info.tx_bits)})")
print(f"ORx {info.orx_rate_khz/1000:.2f} MSPS, {info.rx_bits} bit (FS {full_scale(info.rx_bits)})")

iq = load_tab_iq(SIGNAL_PATH)
i_tx, q_tx = prepare_tx(iq, info.tx_bits)
ref = i_tx.astype(np.float64) + 1j * q_tx.astype(np.float64)
print(f"Signal: {Path(SIGNAL_PATH).name}  N={len(iq)}  PAPR {papr_db(iq):.2f} dB")

## 3. Connect, program, LO1, starting attenuation

In [ ]:
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()

radio.set_tx_atten(TX_CHANNEL, TX_ATTEN_START_DB)
lock = radio.retune_lo("LO1", int(LO1_HZ))

for k, v in verify_status(radio).items():
    print(f"{k}: {v}")
print(f"LO1 = {radio.get_lo('LO1')/1e6:.3f} MHz   lock 0x{lock:X}")
print(f"TX1 atten = {radio.get_tx_atten(TX_CHANNEL):.2f} dB")

## 4. Search and capture at the lock

TX starts, the search runs, and one capture is taken at the lock. TX stops
when this cell ends, whatever happens.

In [ ]:
def show(rec):
    nan = float("nan")
    clip = "  CLIP" if rec.get("pa_clipped") else ""
    print(f"  atten {rec['atten_db']:5.2f} dB  gain {rec['orx_gain']:3d}  "
          f"peak {rec['peak_dbfs']:+5.1f} dBFS  railed {rec['railed']:<4d}  "
          f"PAPR comp {rec.get('papr_compression_db', nan):5.2f}  "
          f"gain comp {rec.get('gain_compression_db', nan):5.2f} dB  "
          f"slope {rec.get('top_slope', nan):5.3f}{clip}  {rec['action']}")

save_dir = Path(SAVE_DIR)
in_path = save_dir / f"{Path(SIGNAL_PATH).stem}_in.txt"
save_dir.mkdir(parents=True, exist_ok=True)
save_aligned_iq(ref, in_path, info.tx_bits)

try:
    transmit_bands(radio, {TX_CHANNEL: iq}, info.tx_bits)
    result = find_compression_point(
        radio, TX_CHANNEL, ORX_CHANNEL, ref,
        rx_bits=info.rx_bits, fs=FS,
        target_compression_db=TARGET_COMPRESSION_DB, lock_on=LOCK_ON, min_top_slope=MIN_TOP_SLOPE,
        start_atten_db=TX_ATTEN_START_DB, atten_min_db=TX_ATTEN_MIN_DB,
        window_s=WINDOW_MS * 1e-3, oversample=OUTPUT_OVERSAMPLE,
        coarse_step_db=COARSE_STEP_DB, fine_step_db=FINE_STEP_DB, comp_tol_db=COMP_TOL_DB,
        target_dbfs=ORX_TARGET_DBFS, tol_up_db=ORX_TOL_UP_DB, tol_down_db=ORX_TOL_DOWN_DB,
        on_step=show,
    )
    print(f"\nconverged: {result.converged}   clip_limited: {result.clip_limited}   {result.reason}")
    print(f"lock {result.final_atten_db:.2f} dB   ORx gain {result.final_orx_gain}   "
          f"{result.lock_on} compression {result.compression_db:.2f} dB   "
          f"(PAPR {result.papr_compression_db:.2f}, gain {result.gain_compression_db:.2f} dB, "
          f"slope {result.top_slope:.3f})")

    point = capture_point(
        radio, ORX_CHANNEL, ref, rx_bits=info.rx_bits, fs=FS,
        bw_hz=BW_MHZ * 1_000_000, oversample=OUTPUT_OVERSAMPLE,
        min_top_slope=MIN_TOP_SLOPE or PA_CLIP_SLOPE,
    )
finally:
    radio.disable_tx()
    print("TX stopped")

name = condition_name(TX_CHANNEL, 0, LO1_HZ, BW_MHZ)
out_path = save_dir / f"{name}_out.txt"
save_aligned_iq(point.y_aligned, out_path, info.rx_bits)
condition = OperatingCondition(
    name=name, signal=Path(SIGNAL_PATH).name, freq_hz=int(LO1_HZ), bw_mhz=BW_MHZ,
    backoff_db=0, locked_atten_db=result.final_atten_db, tx_atten_db=result.final_atten_db,
    orx_gain=result.final_orx_gain, compression_db=result.compression_db,
    converged=result.converged, peak_dbfs=point.clip.peak_dbfs,
    railed=point.clip.railed_samples, in_file=in_path.name, out_file=out_path.name,
    lock_on=result.lock_on, **point.metrics,
)
with ConditionLog(save_dir / CSV_NAME) as log:
    log.append(condition)

m = point.metrics
print(f"\n{name}  -> {out_path}")
print(f"NMSE {m['nmse_db']:.2f} dB   PAPR {m['papr_in_db']:.2f} / {m['papr_out_db']:.2f} dB   "
      f"ACLR {m['aclr_lower_dbc']:.2f} / {m['aclr_upper_dbc']:.2f} dBc")
print(f"peak {point.clip.peak_dbfs:+.2f} dBFS   railed {point.clip.railed_samples}   "
      f"corr {m['corr']:.4f}")
print(f"gain compression {m['gain_compression_db']:.2f} dB   AM/AM top slope {m['amam_top_slope']:.3f}   "
      f"PA clipped {m['pa_clipped']}")

## 5. Search history and the peak window

In [ ]:
hist = result.history
atten = np.array([r["atten_db"] for r in hist])
comp = np.array([r["compression_db"] for r in hist])
papr_comp = np.array([r.get("papr_compression_db", np.nan) for r in hist])
gain_comp = np.array([r.get("gain_compression_db", np.nan) for r in hist])
slope = np.array([r.get("top_slope", np.nan) for r in hist])
gain = np.array([r["orx_gain"] for r in hist])
ok = np.array([r["orx_ok"] for r in hist], bool)

fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
ax[0].axhspan(TARGET_COMPRESSION_DB - COMP_TOL_DB, TARGET_COMPRESSION_DB + COMP_TOL_DB,
              color="g", alpha=0.15, label="tolerance")
ax[0].plot(atten, comp, "o-", c="0.7", label="all tries")
ax[0].plot(atten[ok], comp[ok], "o", c="C0", label=f"ORx leveled ({LOCK_ON}, the lock)")
other, other_name = (papr_comp, "PAPR") if LOCK_ON == "gain" else (gain_comp, "gain")
ax[0].plot(atten[ok], other[ok], "s--", c="C1", alpha=0.6, label=f"{other_name} compression")
ax[0].set_xlabel("TX attenuation (dB)"); ax[0].set_ylabel("compression (dB)")
ax[0].invert_xaxis(); ax[0].legend(fontsize=8); ax[0].grid(True)
ax[1].plot(atten, gain, "o-")
ax[1].set_xlabel("TX attenuation (dB)"); ax[1].set_ylabel("ORx gain index")
ax[1].invert_xaxis(); ax[1].grid(True)
ax[2].plot(atten[ok], slope[ok], "o-")
if MIN_TOP_SLOPE is not None:
    ax[2].axhline(MIN_TOP_SLOPE, c="r", lw=1, label="clip guard")
    ax[2].legend(fontsize=8)
ax[2].set_xlabel("TX attenuation (dB)"); ax[2].set_ylabel("AM/AM top slope")
ax[2].invert_xaxis(); ax[2].grid(True)
plt.tight_layout(); plt.show()

x, y = ref, point.y_aligned
k = int(np.argmax(np.abs(x)))
sl = slice(max(0, k - 100), max(0, k - 100) + 200)
xw, yw = x[sl], y[sl]
yw = yw / (np.vdot(xw, yw) / np.vdot(xw, xw))
fig, ax = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True)
ax[0].plot(np.abs(xw), ".-", label="input"); ax[0].plot(np.abs(yw), ".-", label="output")
ax[0].set_ylabel("codes"); ax[0].set_title("200 samples around the peak")
ax[0].legend(); ax[0].grid(True)
ax[1].plot(np.rad2deg(np.unwrap(np.angle(xw))), ".-", label="input")
ax[1].plot(np.rad2deg(np.unwrap(np.angle(yw))), ".-", label="output")
ax[1].set_xlabel("sample"); ax[1].set_ylabel("deg"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

## Safe-state and disconnect

In [ ]:
radio.safe_state()
radio.disconnect()
print("TX safe, board disconnected")